# Evaluate a fine-tuned SAM2 checkpoint on MSD-US (oil / scratch / stain)

This notebook assumes you have the project files (`data.py`, `sam2_utils.py`, etc.) in the same folder as this notebook.


## 1) Configuration

In [1]:
from pathlib import Path

# Dataset
DATA_DIR        = Path("data/MSD-US/test")  # <-- change if needed
DATASET_FORMAT  = "msd_us"                  # auto | csv | msd_us
GT_DIRNAME      = "ground_truth"
CATEGORIES      = ["oil", "scratch", "stain"]  # None = all categories found

TEST_SIZE       = 0.2      # split ratio (used only if you don't already have a dedicated test set)
RANDOM_SEED     = 42

# Model
SAM2_CHECKPOINT = "sam_trained_models/sam2_hiera_tiny.pt"
MODEL_CFG       = ""       # empty -> infer from checkpoint name
FINETUNED_WEIGHTS = "checkpoints/fine_tuned_sam2_final.torch"  # <-- your output

DEVICE          = "cuda"   # cuda or cpu

# Eval settings
MAX_SIDE        = 1024
NUM_POINTS_PER_INSTANCE = 30
MAX_EVAL_SAMPLES = 200     # set None to evaluate all samples


## 2) Imports + helpers

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from data import load_train_test_data, read_image, resize_to_max_side, get_points
from sam2_utils import build_predictor, load_finetuned_weights

def infer_model_cfg_from_ckpt(ckpt_path: str) -> str:
    name = Path(ckpt_path).name.lower()
    if "tiny" in name or "hiera_t" in name:
        return "sam2_hiera_t.yaml"
    if "small" in name or "hiera_s" in name:
        return "sam2_hiera_s.yaml"
    if "base_plus" in name or "bplus" in name or "b+" in name:
        return "sam2_hiera_b+.yaml"
    if "base" in name or "hiera_b" in name:
        return "sam2_hiera_b.yaml"
    if "large" in name or "hiera_l" in name:
        return "sam2_hiera_l.yaml"
    raise ValueError(f"Cannot infer model_cfg from checkpoint name: {ckpt_path}")

def best_mask_from_multimask(masks: np.ndarray, scores: np.ndarray):
    # masks: (K,H,W) or (K,1,H,W)
    m = masks[:, 0] if masks.ndim == 4 else masks
    s = scores[:, 0] if scores.ndim == 2 else scores
    idx = int(np.argmax(s))
    return (m[idx] > 0).astype(np.uint8), float(s[idx])

def iou_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = pred.astype(bool)
    gt = (gt > 0).astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float(inter / (union + 1e-6))

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred = pred.astype(bool)
    gt = (gt > 0).astype(bool)
    inter = np.logical_and(pred, gt).sum()
    return float(2 * inter / (pred.sum() + gt.sum() + 1e-6))


## 3) Load dataset pairs

In [3]:
categories = CATEGORIES if (CATEGORIES and len(CATEGORIES)) else None

train_data, test_data = load_train_test_data(
    DATA_DIR,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    dataset_format=DATASET_FORMAT,
    gt_dirname=GT_DIRNAME,
    categories=categories,
    verbose=True,
)

pool = test_data if len(test_data) else train_data
print(f"Using pool size = {len(pool)}")
print("Example item:", pool[0])


[MSD-US] Found pairs=1200 (missing masks for 0/1200 images)
Using pool size = 240
Example item: {'image': 'data/MSD-US/test/oil/Oil_0235.jpg', 'annotation': 'data/MSD-US/test/ground_truth/Oil_0235.png', 'category': 'oil'}


## 4) Load model + fine-tuned weights

In [4]:
if not MODEL_CFG:
    MODEL_CFG = infer_model_cfg_from_ckpt(SAM2_CHECKPOINT)

predictor = build_predictor(MODEL_CFG, SAM2_CHECKPOINT, device=DEVICE)
load_finetuned_weights(predictor, FINETUNED_WEIGHTS, device=DEVICE, strict=False)
predictor.model.eval()

print("Loaded model:", MODEL_CFG)


Loaded model: sam2_hiera_t.yaml


## 5) Evaluate IoU / Dice

In [ ]:
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def eval_one(sample):
    image, mask = read_image(sample["image"], sample["annotation"])
    image, mask, _ = resize_to_max_side(image, mask, max_side=MAX_SIDE)

    pts = get_points(mask, num_samples_per_instance=NUM_POINTS_PER_INSTANCE, seed=RANDOM_SEED)
    if pts.shape[0] == 0:
        return None

    with torch.no_grad():
        predictor.set_image(image)
        masks, scores, _ = predictor.predict(
            point_coords=pts,
            point_labels=np.ones((pts.shape[0],), dtype=np.int64),  # (N,) NOT (N,1)
)


    pred, sam_score = best_mask_from_multimask(np.asarray(masks), np.asarray(scores))
    iou = iou_score(pred, mask)
    dice = dice_score(pred, mask)
    return iou, dice, sam_score, image, mask, pred

# Choose subset
idxs = list(range(len(pool)))
random.shuffle(idxs)
if MAX_EVAL_SAMPLES is not None:
    idxs = idxs[: int(MAX_EVAL_SAMPLES)]

ious, dices, sam_scores = [], [], []
examples = []

for k, i in enumerate(idxs, 1):
    out = eval_one(pool[i])
    if out is None:
        continue
    iou, dice, s, image, mask, pred = out
    ious.append(iou); dices.append(dice); sam_scores.append(s)
    if len(examples) < 6:
        examples.append((image, mask, pred, iou, dice, s))
    if k % 25 == 0:
        print(f"{k}/{len(idxs)} -> mean IoU={np.mean(ious):.4f} mean Dice={np.mean(dices):.4f}")

print("Final results:")
print(f"  N={len(ious)}")
print(f"  mean IoU  = {np.mean(ious):.4f}  (median {np.median(ious):.4f})")
print(f"  mean Dice = {np.mean(dices):.4f} (median {np.median(dices):.4f})")


RuntimeError: Tensors must have same number of dimensions: got 3 and 2

## 6) Quick plots

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(ious, bins=20)
plt.title("IoU histogram")
plt.xlabel("IoU"); plt.ylabel("Count")
plt.show()

plt.figure(figsize=(6,4))
plt.hist(dices, bins=20)
plt.title("Dice histogram")
plt.xlabel("Dice"); plt.ylabel("Count")
plt.show()


## 7) Visualize a few examples

In [ ]:
for (image, mask, pred, iou, dice, s) in examples:
    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.title("Image")
    plt.imshow(image)
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.title("GT mask")
    plt.imshow(mask, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title(f"Pred (IoU={iou:.3f}, Dice={dice:.3f})")
    plt.imshow(pred, cmap="gray")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
